# AI 활용에 관하여

과제는 AI 를 사용할 것을 전제로 만들어지고 있습니다.
하지만, 학습의 효율성을 위해서 무분별한 AI 사용은 권장드리지 않고 있어요.

따라서, AI 를 피어리뷰어로서 생각하고 사용해주시는 것을 권고드립니다.  
이를 판단할 수 있는 하나의 기준으로는 같은 수업을 듣는 다른 친구에게 물어볼 수 있는 질문인가? 라고 스스로 물어보는 것도 좋은 기준이 될 수 있을 것 같습니다.

아래는 이에 대한 몇가지 예시입니다.


> AI 사용 가능:
> - 문법 및 API 확인
> - 오류 메시지 해석
> - 실행 환경 문제 해결
> - 내가 작성한 설명의 반례 요청

> AI에 맡기지 않을 것:
> - 코드가 어떻게 동작하는지에 대한 내 설명
> - 실행 결과에서 근거 선택
> - 의견 및 선택의 최종 판단

# 6주차 과제: NLP 활용과 스터디 회고

지난주에는 번역문을 만드는 도중에 원문의 각 위치를 다시 참고할 수 있다는 점을 살펴보았습니다.  
이번에는 그때 사용한 Attention이 **어디를 볼지 정하고, 정보를 모으는 과정** 을 코드로 확인해봅시다.  
그다음에는 1~5주차에 배운 내용을 되짚고, 스터디 전체를 회고해봅시다.

## 목차

1. [6주차 실습: Attention과 Encoder-Decoder](#week6-practice) (Q1~Q3)
2. [1~5주차 핵심 질문 정리](#review-weeks) (Q4~Q8)
3. [블로그 1: 스터디 회고 및 정리](#study-retrospective) (Q9)
4. [블로그 2: 자기주도 학습](#self-directed-study) (Q10)

<a id="week6-practice"></a>
# 1. 6주차 실습: Attention과 Encoder-Decoder

발표에서 본 **"마트에서 배를 샀다"** 를 떠올려봅시다. '배'가 무엇을 뜻하는지 알려면 어떤 단어를 참고하면 좋을까요?  
Attention에서는 참고할 정보를 찾는 쪽을 **Query(Q)**,  
각 토큰이 어떤 질의와 잘 맞는지 비교할 때 쓰는 표현을 **Key(K)**,  
실제로 가져올 정보를 **Value(V)** 로 둡니다.

한 Q와 여러 K의 내적을 스케일링하고 softmax를 적용하면, 각 V를 얼마나 가져올지 정하는 비율을 얻을 수 있습니다.  
그 비율로 V를 가중합해 해당 위치의 새 표현을 만듭니다.  
아래에서는 5주차 번역 예시의 원문 `I like apples`와 번역문 앞부분 `<BOS> 나는`을 작은 벡터로 표현해 이 계산을 살펴봅시다.  

**Q1.** 코드에서 `like`의 표현은 어떻게 만들어지나요?

아래 코드에서는 `query`, `key`, `value`에 모두 `source`를 넣습니다.  
같은 원문으로 Q, K, V를 만들기 때문에 **Self-Attention** 라는 것을 알 수 있습니다.  
`num_heads=2`인 층은 같은 입력을 head마다 따로 투영하고, `return_attention_scores=True`를 주면  
최종 출력과 함께 head별 참조 비율을 돌려줍니다.

아래 셀에서 `like` 한 위치의 내적 점수, softmax를 거친 참조 비율, V의 가중합을 순서대로 확인해보세요.  
`head`를 `0`에서 `1`로 바꾸어 이 셀을 다시 실행하면 어떤 값이 달라지나요?  
층은 각 head의 가중합을 이어 붙이고 출력 투영을 적용해 `self_output`을 만듭니다. 코드의 변수와 출력값을 짚어 설명해주세요.

In [1]:
import numpy as np
from tensorflow import keras

keras.utils.set_random_seed(42)

source_tokens = ["I", "like", "apples"]

# 축 순서: [배치, 토큰, 벡터 성분]
source = np.array([
    [  # 첫 번째 문장
        [2, 0, 0, 1],  # I
        [1, 0, 1, 0],  # like
        [0, 1, 2, 0],  # apples
    ]
], dtype="float32")

attention = keras.layers.MultiHeadAttention(num_heads=2, key_dim=2)
head = 0  # 1로 바꿔서 이 셀을 다시 실행해보세요.

self_output, self_weights = attention(
    query=source, key=source, value=source,
    return_attention_scores=True, training=False,
)

print("참조 비율의 shape [배치, head, query, key]:", self_weights.shape)
print("행과 열의 토큰 순서:", source_tokens)
print(f"Self-Attention (head {head})")
print(np.round(self_weights[0, head].numpy(), 3))

# 선택한 head에서 like의 Q와 모든 원문 토큰의 K, V를 꺼냅니다.
like_index = source_tokens.index("like")
like_query = attention.query_dense(source).numpy()[0, like_index, head]
keys = attention.key_dense(source).numpy()[0, :, head]
values = attention.value_dense(source).numpy()[0, :, head]
like_scores = keys @ like_query / np.sqrt(like_query.size)
like_weights = self_weights[0, head, like_index].numpy()
like_context = like_weights @ values

print("like의 스케일링한 내적 점수:", np.round(like_scores, 3))
print("like의 softmax 참조 비율:", np.round(like_weights, 3))
print("선택한 head에서 like의 V 가중합:", np.round(like_context, 3))
print("전체 Attention 출력 shape:", self_output.shape)

참조 비율의 shape [배치, head, query, key]: (1, 2, 3, 3)
행과 열의 토큰 순서: ['I', 'like', 'apples']
Self-Attention (head 0)
[[0.808 0.169 0.023]
 [0.277 0.378 0.345]
 [0.032 0.257 0.711]]
like의 스케일링한 내적 점수: [-0.141  0.173  0.08 ]
like의 softmax 참조 비율: [0.277 0.378 0.345]
선택한 head에서 like의 V 가중합: [0.103 0.598]
전체 Attention 출력 shape: (1, 3, 4)


**>>> like는 [1,0,1,0]로 표현됩니다.
head0에서 head1로 바뀌면 내적점수, 참조비율과 가중합이 달라지게 됩니다. head 0 like context는 [0.103, 0.598], head 1 like context는 [-0.275, 0.872]이고 층은 각 head의 가중합을 이어붙이고 출력 투영을 적용해 self output을 만듭니다. 이를 I와 apples에도 적용해 (1,3,4)라는 출력 shape이 나옵니다.<<<**

**Q2.** Causal mask와 Cross-Attention은 코드의 어느 부분에서 결정되나요?

스터디에서는 다음 토큰을 예측하는 위치가 오른쪽에 있는 정답 토큰을 미리 볼 수 없도록 **causal mask**를 적용한다고 했고,  
질문과 이에 대한 정보가 떨어져있을 때는 토큰 배열에서 Q를, 다른 토큰 배열에서 K와 V를 만드는 **Cross-Attention**을 사용한다고 했습니다.

아래 셀은 같은 `attention` 층으로 기본 Self-Attention, causal mask를 적용한 Self-Attention, Cross-Attention을 계산합니다.  
앞의 두 결과를 비교해 `use_causal_mask=True`가 `like` 행에서 어느 값을 바꾸는지 확인하고, 그 위치가 왜 가려지는지 설명해보세요.  
기본 Self-Attention과 Cross-Attention을 비교하면 Q의 입력만 `prefix`로 바뀝니다.  
출력된 행렬의 행과 열이 각각 어떤 문장과 대응하는지 코드를 따라가면서 확인해봅시다.


참고자료  
[Keras MultiHeadAttention 문서](https://keras.io/api/layers/attention_layers/multi_head_attention/)의 `query`, `value`, `use_causal_mask`

In [2]:
import numpy as np
from tensorflow import keras

keras.utils.set_random_seed(42)

source_tokens = ["I", "like", "apples"]
prefix_tokens = ["<BOS>", "나는"]

# 축 순서: [배치, 토큰, 벡터 성분]
source = np.array([
    [  # 첫 번째 문장
        [2, 0, 0, 1],  # I
        [1, 0, 1, 0],  # like
        [0, 1, 2, 0],  # apples
    ]
], dtype="float32")
prefix = np.array([[[1, 1, 0, 0], [2, 0, 1, 0]]], dtype="float32")

attention = keras.layers.MultiHeadAttention(num_heads=2, key_dim=2)
head = 0

_, self_weights = attention(
    query=source, key=source, value=source,
    return_attention_scores=True, training=False,
)
_, masked_weights = attention(
    query=source, key=source, value=source,
    use_causal_mask=True, return_attention_scores=True, training=False,
)
_, cross_weights = attention(
    query=prefix, key=source, value=source,
    return_attention_scores=True, training=False,
)

print("행과 열:", source_tokens)
print(f"Self-Attention (head {head})")
print(np.round(self_weights[0, head].numpy(), 3))
print(f"Causal Self-Attention (head {head})")
print(np.round(masked_weights[0, head].numpy(), 3))

print()
print(f"Cross-Attention (head {head})")
print("행:", prefix_tokens, "/ 열:", source_tokens)
print(np.round(cross_weights[0, head].numpy(), 3))

행과 열: ['I', 'like', 'apples']
Self-Attention (head 0)
[[0.808 0.169 0.023]
 [0.277 0.378 0.345]
 [0.032 0.257 0.711]]
Causal Self-Attention (head 0)
[[1.    0.    0.   ]
 [0.422 0.578 0.   ]
 [0.032 0.257 0.711]]

Cross-Attention (head 0)
행: ['<BOS>', '나는'] / 열: ['I', 'like', 'apples']
[[0.522 0.389 0.089]
 [0.488 0.373 0.139]]


**>>> use_casual_mask=True를 적용하면 [0.422 0.578 0.000]으로 변경됩니다. like 오른쪽에 있는 apples가 미래 토큰이므로 다음 토큰을 예측할 때  미래의 정답 토큰을 참조하는 것을 방지할 수 있습니다. self-attention에서는 Q, K, V 모두 source에서 생성하므로 행과 열이 모두 ['I', 'like', 'apples']에 대응합니다. cross-attention에서는 Q만 prefix에서 생성하고 K와 V는 source에서 생성하므로 행은 ['<BOS>', '나는'], 열은 ['I', 'like', 'apples']에 대응합니다. <<<**

**Q3.** Encoder가 원문을 읽은 결과는 Decoder에서 어떻게 쓰이나요?

이제 스터디의 Encoder-Decoder 그림처럼 세 Attention을 연결해봅시다.  

Encoder는 Self-Attention으로 원문 각 위치의 표현을 만듭니다.  
Decoder는 먼저 **causal self-attention**으로 지금까지의 번역문을 읽고,  
**cross-attention**에서 Encoder가 만든 원문 표현을 참고합니다.  
각 경로에 기존 입력 표현을 더해주는 연결은 4주차의 residual connection을 떠올리면 됩니다.

`encode(source)`가 반환한 `encoded_source`를 따라가며 Decoder의 어느 인자로 전달되는지 확인해보세요.  
같은 원문을 두고 Decoder에 `<BOS>`만 넣은 경우와 `<BOS> 나는`을 넣은 경우를 비교해봅시다.  
두 입력에서 공통으로 등장하는 첫 토큰 `<BOS>` 에 대해서 `나는` 이 뒤에 추가되어도  
첫 위치의 결과가 같은 이유를 코드에서 찾아 설명해주세요.  

마지막으로 `output_layer`와 `softmax`를 거친 출력의 두 행이 각각 무엇을 뜻하는지 적어주세요.  

참고자료  
[Keras 번역 예제](https://keras.io/examples/nlp/neural_machine_translation_with_transformer/#building-the-model)

In [3]:
import numpy as np
from tensorflow import keras

keras.utils.set_random_seed(42)

# 축 순서: [배치, 토큰, 벡터 성분]
source = np.array([
    [  # 첫 번째 문장
        [2, 0, 0, 1],  # I
        [1, 0, 1, 0],  # like
        [0, 1, 2, 0],  # apples
    ]
], dtype="float32")
prefix = np.array([[[1, 1, 0, 0], [2, 0, 1, 0]]], dtype="float32")

encoder_self = keras.layers.MultiHeadAttention(num_heads=2, key_dim=2)
decoder_self = keras.layers.MultiHeadAttention(num_heads=2, key_dim=2)
decoder_cross = keras.layers.MultiHeadAttention(num_heads=2, key_dim=2)

def encode(source_vectors):
    return source_vectors + encoder_self(query=source_vectors, value=source_vectors)

def decode(prefix_vectors, encoded_source):
    context = prefix_vectors + decoder_self(
        query=prefix_vectors, value=prefix_vectors, use_causal_mask=True,
    )
    return context + decoder_cross(query=context, key=encoded_source, value=encoded_source)

encoded_source = encode(source)
decoded_prefix = decode(prefix, encoded_source)
decoded_bos_only = decode(prefix[:, :1], encoded_source)

vocab = ["나는", "사과를", "좋아한다", "<EOS>"]
output_layer = keras.layers.Dense(len(vocab))
next_token_probs = keras.ops.softmax(output_layer(decoded_prefix), axis=-1)

print("Encoder 출력:", encoded_source.shape)
print("Decoder 출력:", decoded_prefix.shape)
print("다음 토큰 확률 [배치, 번역문 위치, 단어장]:", next_token_probs.shape)
print("단어장 순서:", vocab)
print(np.round(next_token_probs[0].numpy(), 3))

# 두 입력 모두 첫 토큰은 <BOS>입니다.
bos_only = decoded_bos_only[0, 0].numpy()
bos_with_next = decoded_prefix[0, 0].numpy()
print("같은 원문에서 <BOS> 위치의 Decoder 출력 비교")
print("<BOS>만 입력:", np.round(bos_only, 3))
print("<BOS> 나는 입력:", np.round(bos_with_next, 3))
print("두 벡터가 같은가요?", np.allclose(bos_only, bos_with_next, atol=1e-6))

Encoder 출력: (1, 3, 4)
Decoder 출력: (1, 2, 4)
다음 토큰 확률 [배치, 번역문 위치, 단어장]: (1, 2, 4)
단어장 순서: ['나는', '사과를', '좋아한다', '<EOS>']
[[0.332 0.592 0.047 0.029]
 [0.185 0.799 0.011 0.005]]
같은 원문에서 <BOS> 위치의 Decoder 출력 비교
<BOS>만 입력: [ 1.38  -1.298  0.602  0.736]
<BOS> 나는 입력: [ 1.38  -1.298  0.602  0.736]
두 벡터가 같은가요? True


**>>> encoded_source는 decoder의 Cross-Attention에서 Key와 Value로 사용됩니다. decoder에 BOS만 입력한 경우와 BOS 나는을 입력한 경우 첫 번째 위치의 출력은 동일합니다. decoder_self에서 use_casual_mask=True를 적용했기 때문입니다. BOS는 뒤에 추가된 '나는'을 참조할 수 없기 때문에 두 입력에서 첫 번째 위치의 context는 동일합니다. out_layer는 decoder의 출력을 단어장에 있는 각 토큰의 점수로 변환하고, softmax는 이를 확률분표로 변환합니다. 첫 번째 행은 BOS 다음에 올 토큰의 확률 분포이고, 두 번째 행은 BOS 나는 다음에 올 토큰의 확률분포입니다. <<<**

<a id="review-weeks"></a>
# 2. 1~5주차 핵심 질문 정리

이제 처음으로 돌아가서, 우리가 어떤 문제를 만날 때마다 모델의 구조나 쓰는 방법을 어떻게 바꾸어 왔는지 살펴봅시다.  
지난 과제에서 직접 넣어본 입력이나 기억나는 결과를 떠올리면서 배웠던 내용과 궁금한 점 등 모두 정리해봅시다.  

Keras 이야기는 많이 하진 않겠습니다. 개념을 알고있다면 개념에 해당하는 Keras Class 는 찾으면 그만이기 때문입니다.  

### **Q4.** [1주차 모델 추론](../week1/week1.ipynb)  

AI 모델이라는 것은 결국 커다란 하나의 함수라고 이야기했습니다.  
즉, 모델에 질의응답을 하는 것은 질문이라는 입력을 바탕으로 모델이라는 함수가 처리하여, 답변이라는 출력을 반환하는 것입니다.  
가장 작은 최소 단위의 계산은 퍼셉트론으로 여러 계산을 묶어서 한번에 행렬곱(정확히는 Affine Transform) 을 수행할 수 있다는 점을 알 수 있었습니다.  

TMI 로는, Affine Transform 이 무엇인지, 그게 행렬곱과 왜 동일하게 취급하는지가 알고싶다면 다음의 자료를 참고해주세요.  
Affine Transform 이란 y =$W$*$x$ + $b$ 의 행렬곱 이후에 편향을 더하는 연산까지 모두 묶은 것을 의미합니다.  
엄밀하게 선형변환은 원점을 보존해줘야하기 때문에 $+b$ 는 선형변환은 아닌 것처럼 보일수도 있습니다.

https://angeloyeo.github.io/2024/06/28/Affine_Transformation.html

**>>> 퍼셉트론의 구조 wx + b = z는 가중치로 입력의 중요도를 결정하고 편향으로 어느 지점으로 입력을 시작할지 조절하는 역할을 합니다. sigmoid는 한 후보가 정답일 가능성을 구할 때 사용하고 softmax는 여러 후보의 확률을 구할 때 사용합니다.<<<**

### **Q5.** [2주차 모델 학습](../week2/week2.ipynb)

"모델이 학습한다" 라는 말이 구체적으로 무엇인가? 에 대해 다루어보았습니다.  
Cost Function 이 학습의 목표, Gradient Descent 가 해당 목표를 바탕으로 학습을 진행하는 방법론이었습니다.  
그 외 구체적인 방법에 따라 다양한 하이퍼파라미터가 등장했습니다.  

Gradient 를 구하기 위해서는 편미분, chain rule 등을 활용했고, chain rule 덕분에 Back Propagation 이 가능했습니다.
Back Propagation 이 일종의 Dynamic Programming(알고리즘에 나오는 그것이 맞습니다) 이라는 점도 흥미로운 점입니다.  

1주차와 내용이 걸쳐있지만, AI 모델에서 선형성(Linearity) 의 유무가 어떤 영향이 있는지도 살펴보는 것도 흥미롭습니다.  
간단하게 말하면, 선형성이 존재한다는 것은 $f(x + y) = f(x) + f(y)$ 를 만족한다는 의미이기 때문에 Layer 를 쌓는 것을 무의미하게 만듭니다.  
이 때문에 비선형성을 가진 활성화함수를 활용하게 됩니다. 활성화함수가 모두 비선형성인 이유가 여기에 있죠.  
아래 블로그의 활성화함수 파트에서 가볍게 다루어보았으니 한번 살펴보는 것도 좋겠습니다.

https://open-alpaca-lab.tistory.com/3  

대놓고 선형성을 다뤄보기도 했으나... 난이도가 조금 있습니다.  
https://open-alpaca-lab.tistory.com/18

**>>> 과적합이 발생해서 일반화 성능이 낮아지는 것을 막기 위해 L1 regulation과 L2 regulation, dropout을 사용합니다. L1 regulation은 가중치의 절댓값을 loss에 추가하여 일부 가중치를 0으로 만들고, L2 regulation은 가중치의 제곱을 loss에 추가하여 전체적으로 가중치가 지나치게 커지는 것을 억제합니다. dropout은 학습할 때 일부 신경망의 뉴런을 확률적을 비활성화 시켜 특정 뉴런에 지나치게 의존하는 것을 방지해 과적합을 막습니다. <<<**

### **Q6.** [3주차 CNN 기초](../week3/week3.ipynb)

이미지라는 도메인에 특화한다면 어떤 구조를 활용해볼 수 있을까? 에 대해 다루어보았습니다.  
이미지가 어떤 특성을 가지는지 살펴보고, 그러한 특성을 모델 구조에 활용하는 Inductive Bias 를 알아보았습니다.  
Convolution, Pooling 이 어떤 연산이고, 왜 사용하는가? 에 대해도 살펴보았습니다.  

개인적으로는 Convolution 이라는 연산이 그리 편하게 다가오진 않습니다.  
그럼에도 Convolution 이라는 연산은 이미지 처리에, 특히 신호 처리에 핵심이 되는 연산 중에 하나입니다.  
정말 많은 자료와 이론이 이 개념이 중요하다는 사실을 반증하죠.  

이번 스터디와는 무관하기 때문에 다루지 않았습니다만, 이미지 처리가 사실 신호 처리와 유사하다는 점을 한번쯤은 공부하시면 좋을 듯합니다.  
어떻게 이미지가 $sin, cos$ 으로 표현할 수 있는지,  
Fourier Transform 이 무엇인지,  
Filter 로 불투명하게, 대비가 확실하게 만드는게 무엇인지  
등을 알아볼 수 있습니다.  

https://visionbook.mit.edu/part_linear_filters.html  
https://visionbook.mit.edu/image_processing_fourier.html  
https://angeloyeo.github.io/2019/06/18/Discrete_Time_Convolution.html  

**>>> convolution은 입력 데이터에 kernel을 이동시키면서 겹치는 값끼리 곱한 후 합산합니다. 이미지의 픽셀간의 관계를 파악하고 특성을 추출해야하기 때문입니다. pooling은 feature map의 일정 영역에서 대표값을 선택하여 크기를 줄이는 연산입니다. Max pooling은 최댓값을 선택하고 average pooling은 평균값을 계산합니다. <<<**

### **Q7.** [4주차 CNN 활용](../week4/week4.ipynb)

CNN 의 활용 모델인 ResNet 에 집중해보았습니다. 동시에 모델이 모델에게 지식을 전달해주는 Transfer Learning 에 대해 알아보았습니다.  
Residual Block, Bottleneck Block, GAP, Batch Normalization, Fine Tuning, Data Augmentation, Distillation 에 대해 알아보았습니다.

그래놓고 첫 질문은 Universal Approximation Theory 였죠 ㅋㅋ;  
사실 2주차에 넣을까하다가 넣지 못한게 아쉬워 조금 질척여봤습니다.  

그리고 ResNet 의 핵심개념 중에 자연어처리와 연결되는 부분이 있다고 했죠.  
그 부분도 자연어처리를 배운 지금, 다시 한번 살펴보는 것도 좋겠습니다. 하지만 여전히 어렵긴 할 것입니다.  
관련해서 세미나로 몇번 다루긴 했는데, 참고해셔도 좋을 듯합니다.

https://youtu.be/7LVRciBEGaM?si=ZRh8R0UeAifz_bZN  
https://youtu.be/eRz3s_nb9zY?si=a53EnnIdu87lbT3J - 이 영상은 이야기하고자 한 부분에 대한 후속 응용 내용이긴합니다. 가볍게 봐주세요.

모델의 주요 연산이 어떤 특징을 가지고 있는지 왜 사용하는지도 잘 정리해봅시다.
Normalization 은 입력에 대한 전처리로서, Batch 데이터에 대해, Feature 에 대해 등 다양한 곳에서 사용되니 잘 구분해주세요.  

그 다음으로는 Transfer Learning 으로서, Mapping 으로서, Input 으로서, Output 으로서 전달하는 방법에 대해 알아보았습니다.
사실 프롬프팅도 일종의 Transfer Learning 로 볼수도 있긴하다고 합니다. 자세한 내용은 아래의 링크를 참고해보세요.

https://visionbook.mit.edu/transfer_learning.html#domain-adaptation

마지막에 Representation Learning 에 대해서도 다뤄보았는데,  
모델이라는 것이 결국 입력과 출력에 대한 Mapping, 함수라는 점, 유용한 중간표현을 쓰겠다는 점에 집중하면 조금 이해가 쉬울 것 같습니다.

https://visionbook.mit.edu/representation_learning.html

**>>> ResNet은 깊은 신경망에서 층이 깊어질수록 학습 성능이 저하되는 문제를 완화하기 위해 Residual Connection을 사용하는 CNN모델입니다. Residual Connection은 입력 x를 여러 층 연산 결과 F(x)에 직접 더하여 계산합니다. F(x) = 0을 학습할 때 불필요한 학습을 할 필요가 없기 때문에 degradation problem을 완화할 수 있습니다. <<<**

### **Q8.** [5주차 NLP 기초](../week5/week5.ipynb)

자연어처리란 무엇인가? 자연어처리를 어떻게 할 것인가? 에 대한 역사적인 내용을 학습했습니다.  
이 과정에서 어떤 한계가 있었는지, 이를 어떤 관점으로 극복하고자 했는지 등을 살펴보았습니다.  

결국 AI 는 "어떻게 숫자로 변환할 것인가?", "그 숫자를 어떻게 계산하고 처리할 것인가?" 에 대한 것이라는 점을  
이미지와 자연어에서 조금은 느끼신다면 좋을 듯합니다.  
특히, Transformer 라는 구조가 자연어뿐만 아니라, ViT 라는 이름으로 이미지 분야에 널리 사용하는 것에서도 알 수 있습니다.  

스터디에서는 자연어 분야에 맞게 다른 자료를 바탕으로 준비했지만, 좀 더 포괄적인 관점에서 파악하고 싶다면 아래의 자료도 추천드립니다. 

https://visionbook.mit.edu/transformers.html#sec-transformers-ViT_arch

자연어분야에서 더 학습하고 싶다면, 그 유명하나 CS224N 도 있죠.

https://youtu.be/fyc0Jzr74y4?si=YQ_E9KjrZEaCaCVQ - 여기서 시작해서 9강까지 본다면, 6주차까지의 내용을 다루게 될 것입니다.

조금더 가볍게, 그리고 풍부한 시각화를 원하신다면 아래의 영상도 있습니다.

https://youtu.be/LPZh9BOjkQs?si=v6f776GXjyovA9_W - 여기서부터 Ch.7 까지 보시면 됩니다. (근데 그냥 다 보세요.)

**>>> LSTM과 GRU는 gate를 사용해 중요한 정보를 유지하고 불필요한 정보를 제거함으로써 장기 의존성 문제를 완화했습니다. attention은 현재 단어와 관련성이 높은 토큰에 더 큰 가중치를 부여하는 방법입니다. Transformer는 Self-Attention을 이용해 토큰 사이의 관계를 계산하며 여러 토큰을 병렬로 처리할 수 있도록 했습니다.  <<<**

<a id="study-retrospective"></a>
# 3. 블로그 1: 스터디 회고 및 정리

앞에서 진행한 복습한 내용은 어땠나요?  
서로 어떻게 연결되어있는지, 각 내용이 무엇을 다루고자 했는지, 왜 사용하고자 하는지 등이 정리가 될까요?  
스터디를 마무리한다는 의미에서 전체 주차에 대해 정리해봅시다.

**Q9.** 스터디를 시작하기 전의 자신을 떠올리며 1~6 주차의 내용을 정리해주세요.

각 주차가 전달하고 싶은 핵심적인 내용이 무엇이었나요? 이것만은 잊지 않고 기억하고 싶은 내용은 무엇인가요?  
각 주차를 했을 때와 지금 다시 살펴봤을 때, 어떻게 다르게 보이나요?  

각 주차의 흐름을 정리해보고, 그 흐름이 전체 주차에서 어떻게 연결되는지도 정리해봅시다.

아래는 도움이 될법한 제 의견을 달아두었습니다. 참고만 해주세요.  
무엇을 풀고자 했는지 문제를 이해했나요? 그 문제를 어떻게 해결하고자 했나요? 핵심 아이디어가 무엇인가요?  
어떤 상황에서 스스로 이 방법을 사용해야겠다고 느낄만한 포인트는 무엇인가요?  
계산 그 자체가 아니라 직관과 프레임, 개념에 집중해봅시다.  
다른 주차와 연결되는 부분이 있다면 그것 역시 좋습니다.

위에서 언급한 내용은 제가 공부할 때 역시 중요하게 여기는 포인트인데 아래 글의 일부를 참고하면 도움이 될지도 모르겠습니다.  
[제가 작성한 Random Walk & Heat Equation](https://open-alpaca-lab.tistory.com/23) 의 도입과 마지막 `Recap`을 읽어보면,  
궁금한 점에서 시작해 직접 비교한 뒤 무엇을 이해하게 되었는지 적는 흐름을 볼 수 있습니다. (그 중간은 굳이 안 읽으셔도 됩니다.)

블로그로 남긴다면 링크를, 아니라면 아래에 1차적으로 작성하고, WIL 을 작성할 때 보충해서 정리해주세요.

**>>> 신경망의 개념부터 우리가 사용하는 chat gpt, gemini까지 발전과정을 다룬 것 같습니다. 신경망의 추론과 학습원리, CNN, ResNet, 자연어 처리에서 RNN의 한계를 개선한 LSTM, GRU, Attention을 학습했습니다. Transformer를 이용해 토큰 사이의 관계를 계산하고 다음 토큰을 예측하여 새 문장을 생성하는 원리를 학습하였습니다. 6주동안 학습한 이미지와 자연어를 숫자로 표현하고 처리하는 기술이 현대 생성형 ai의 기반이 될 것 같다는 생각을 했습니다. <<<**

<a id="self-directed-study"></a>
# 4. 블로그 2: 자기주도 학습

**Q10.** 스터디를 진행하면서 **조금 더 알아보고 싶었던 질문 하나**를 골라 별도의 글로 써주세요.

과제에 있던 TMI, 추가 자료 혹은 스스로 품었던 의문 모든 좋습니다.  
스터디 시간에, 과제에 적혀있지 않은 내용을 스스로 정리해주세요.

어느 부분에서 궁금해졌는지, 왜 그 내용을 적기로 했는지, 확인하기 위해 무엇을 찾아봤는지,  
그걸 모르는 사람에게 문제부터 해결까지 설명한다고 했을 때 어떻게 할지 등을 정리해주세요.  
답이 없는 문제거나, 어렵다면 그런 상태로 냅두고 한 노력을 정리해주시면 됩니다.

블로그로 남긴다면 링크를, 아니라면 아래에 1차적으로 작성하고, WIL 을 작성할 때 보충해서 정리해주세요.

**>>> 비선형성을 추가할 때 추가하는 방법이 여러가지가 있는데 하나로 결정되는 이유가 궁금했습니다. 예를 들어 RNN에서 왜 arctan가 아닌 tanh를 사용하는지같은 것이 궁금했습니다.<<<**

# 수고하셨습니다!

아래는 끝난 기념으로 개인적으로 남기는 TMI, 뒷이야기, 의도 등등 입니다. 편하게 읽어주세요.

여기까지 과제도 모두 마무리한(혹은 하고 있는) 여러분 모두 수고 많으셨습니다!  
8월 중순에 처음 회의를 진행했으니, 저에게는 3달의 여정이었네요.  

저는 제가 아는 내용 혹은 공부한 내용을 남과 공유하는 것을 좋아합니다.  
이야기하는 과정에서 제 빈틈을 찾는 것도 있지만, 제가 생각하지 못한 새로운 관점을 들을 수 있는 좋은 기회이기도 하기 때문이죠.  

아마 과제나 제 스터디는 조금 낯설기도 했을 겁니다.  
되도록이면 한번 사용될 지식이 아니라, 감히 앞으로 계속해서 떠오를 **"프레임"** 을 만들고자 했기 때문일 것입니다.  
그 때문에 구체적인 내용보다는 의도, 관점, 이유 등에 집중해서 필요할 때 사용할 수 있는 형태로 남을 수 있도록 노력했습니다.  

처음 시작할 때는 조금 가벼운 마음이었습니다.  
1, 2주차가 포괄적인 내용이니 가장 어렵고, 뒤로 갈수록 쉽겠지? 라는 막연한 생각을 가졌으나  
이런, 뒷 내용도 포괄적으로 이야기해버렸군요... 결국 난이도 조절 실패하고 졸지에 거짓말쟁이가 되었답니다 허허  

저 역시 스터디를 진행하면서 성장할 수 있었습니다.  
이런 난이도 예측에 실패도 해보고, 급하게 마감을 치기도 하고(오늘도), 아는 내용인줄 알았으나 생각보다 모르는게 많다는 점도 발견하고  
제 지식?적으로도 많이 배우게 되었습니다. 덤으로 AI 사용법도 다시 다듬는 기회가 되었고요.  

아마 특이했던 점 중에 하나가 AI 사용법에 대한 지속적인 강조였을 거라고 생각합니다.  
네, 과제에 맨날 있는 "AI 활용에 관하여" 부분입니다.  

저 멘트 자체는 [어떤 강의 초반](https://youtu.be/6FkRvTtUc-o?si=vfFGTiubTeBRXHUK&t=602)에 등장했습니다.  
LLM 을 인간 리뷰어로 생각하고 다뤄라. 옆의 친구에게 똑같이 물어볼 수 있는가? 를 생각해봐라 등  
제 생각과 어느정도 겹치면서, 꽤 명쾌하다고 생각해서 마구 인용했습니다.

저는 2025년 7월 네이버부스트캠프에서 처음으로 Gemini CLI 를 통해 본격적으로 사용하게 되고,  
거기서 실패를 겪고부터 지속적으로 AI 사용법에 대해 고민해왔습니다.  

정말 웹에서 채팅으로만 사용하기도, CLI, MCP, 프로젝트 세팅 등 깊이 사용해보기도 했습니다.  
그런 경험을 바탕으로 학습이나 개인의 성장에 있어서는 명확한 제한이 필요하다는 생각에 이르게 됩니다.  

특히, 개인적인 커리큘럼으로서 LLM 에게 시켜보기도 했으나, 결국 하나의 강의를 듣는 것만 못하다는 결론에 이르게 됩니다.  
강의 커리큘럼에 대한 사람의 깊은 의도를 아직 반영하지는 못한다는 생각입니다.  
심지어, 제가 만들고자 한 과제의 의도도 모두 반영해주지 못했으니까요.

또한, LLM **"만"** 으로는 개인의 생각의 한계를 돌파하기도 많이 어렵겠다고 느끼게 되었습니다.  
되도록이면 저와 스터디를 진행하신 분들은 LLM 에 과하게 의존하지 않는, LLM 없이도 살 수 있는 그런 분들이 되었으면 하는 바람입니다.

자세한 내용은 블로그로 몇번 남겨서 여기서 줄이겠습니다. 

https://open-alpaca-lab.tistory.com/21  
https://open-alpaca-lab.tistory.com/22  
https://open-alpaca-lab.tistory.com/24  

생각이나 의견, 궁금한 점은 언제든 환영이니 편하게 연락주시면 감사하겠습니다!  
저와의 인연은 일단은 여기까지이지만, 제가 여러분에게 작더라도 좋은 영향을 끼쳤으면 좋겠네요.  
다들 정말 수고 많으셨습니다.